# Visualização com t-SNE

Neste notebook vamos carregar as matrizes Bag of Words e TF-IDF geradas no notebook introducao_pln e aplicar o t-SNE para reduzir a dimensionalidade a duas dimensoes. Assim conseguimos visualizar como as noticias se agrupam segundo cada representacao.

## t-SNE

O t-SNE (t-distributed Stochastic Neighbor Embedding) e uma tecnica de reducao de dimensionalidade nao linear, muito usada para visualizacao. Ele tenta colocar pontos parecidos no espacao original proximos no espaco reduzido (em geral 2D), preservando a estrutura local dos dados.

### carregando bases

In [24]:
import pandas as pd
import plotly.express as px
from sklearn.manifold import TSNE

In [25]:
from pathlib import Path
pasta_dados = Path("data")
pasta_dados.mkdir(exist_ok=True)

df_bow = pd.read_csv(pasta_dados/"bow.csv")
df_tfidf = pd.read_csv(pasta_dados/"tfidf.csv")

print(f"BoW:    {df_bow.shape}")
print(f"TF-IDF: {df_tfidf.shape}")
df_bow.head()

BoW:    (323, 3534)
TF-IDF: (323, 1328)


,titulo,subtitulo,descricao,temas,data,ano_mes,ano,mes,hora,turno,...,bow_tema_humanos,bow_tema_infraestrutura,bow_tema_meio,bow_tema_minas,bow_tema_pesquisa,bow_tema_saude,bow_tema_seguranca,bow_tema_social,bow_tema_trabalho,bow_tema_turismo
0,Minas promove capacitação de clubes de futebol...,Fale Agora será apresentado em times femininos...,Em continuidade à propagação e divulgação do P...,"['Social', 'Esportes']",2024-04-27 12:20:00-03:00,2024-04,2024,4,12,Tarde,...,0,0,0,0,0,0,0,1,0,0
1,Minas Gerais fecha primeiro bloco dos Jogos Es...,"Competição acontece em Uberlândia, no Triângul...",A delegação de Minas Gerais encerrou o primeir...,"['Social', 'Esportes']",2025-10-14 15:20:00-03:00,2025-10,2025,10,15,Tarde,...,0,0,0,0,0,0,0,1,0,0
2,Estudantes de BH vão carregar bandeiras de Bra...,"Ao todo, 26 alunos da Escola Estadual Tomás Br...",Esta terça-feira (2/7) é de expectativa para 2...,"['Educação', 'Esportes']",2019-07-02 17:02:00-03:00,2019-07,2019,7,17,Tarde,...,0,0,0,0,0,0,0,0,0,0
3,Governo publica edital 2019 do programa Bolsa ...,As inscrições terminam no próximo dia 12 de se...,A Secretaria de Estado de Desenvolvimento Soci...,"['Social', 'Esportes']",2019-08-27 14:35:00-03:00,2019-08,2019,8,14,Tarde,...,0,0,0,0,0,0,0,1,0,0
4,Cruzeiro e Minas Arena assinam acordo mediado ...,Documento foi assinado na tarde desta sexta-fe...,O acordo comercial entre a Minas Arena e o Cru...,['Esportes'],2023-04-28 18:25:00-03:00,2023-04,2023,4,18,Noite,...,0,0,0,0,0,0,0,0,0,0


### separando metadados e matrizes

In [26]:
colunas_bow = [c for c in df_bow.columns if c.startswith("bow_")]
colunas_tfidf = [c for c in df_tfidf.columns if c.startswith("tfidf_")]

X_bow = df_bow[colunas_bow].values
X_tfidf = df_tfidf[colunas_tfidf].values

titulos = df_bow["titulo"].tolist()

print(f"X_bow:   {X_bow.shape}")
print(f"X_tfidf: {X_tfidf.shape}")

X_bow:   (323, 3519)
X_tfidf: (323, 1313)


### aplicando o t-SNE

In [27]:
perplexity = 8
random_state = 42

tsne_bow = TSNE(n_components=2, perplexity=perplexity, random_state=random_state, init="random")
embed_bow = tsne_bow.fit_transform(X_bow)

tsne_tfidf = TSNE(n_components=2, perplexity=perplexity, random_state=random_state, init="random")
embed_tfidf = tsne_tfidf.fit_transform(X_tfidf)

print("embed_bow:  ", embed_bow.shape)
print("embed_tfidf:", embed_tfidf.shape)

embed_bow:   (323, 2)
embed_tfidf: (323, 2)


### visualizando lado a lado

In [28]:
def plotar(embed, df_meta, nome):
    df_plot = pd.DataFrame({
        "x": embed[:, 0],
        "y": embed[:, 1],
        "indice": df_meta.index,
        "titulo": df_meta["titulo"],
        "data": df_meta["data"],
    })
    fig = px.scatter(
        df_plot,
        x="x",
        y="y",
        hover_data={"indice": True, "titulo": True, "data": True, "x": False, "y": False},
        title=f"t-SNE - {nome}",
        width=700,
        height=700,
    )
    fig.write_html(nome + ".html")
    fig.show()


plotar(embed_bow,   df_bow,   "Bag of Words")
plotar(embed_tfidf, df_tfidf, "TF-IDF")